# 02 · Light tables

**Purpose.** Look at the S2 tables used by detsim, and predict the fiber/PMT S2 ratio from them before looking at any simulation output.

**Inputs.** fibers: `20250408_s2_table.h5`; PMTs: `$ICDIR/database/test_data/NEXT100_S2_LT.h5`; S1: `NEXT100_S1_LT.h5` (PMTs, also used as placeholder for fibers).

**Outputs.** LCE maps (plots).

**What is checked.** table provenance (light yield, statistics), normalization (per EL photon), central LCE for fibers and PMTs and their ratio.

**Next step.** `03_detsim`.

In [ ]:
# Setup: run `source env.sh` before starting Jupyter, or pick the IC-3.8-2024-06-08 kernel.
import os, sys, glob
sys.path.insert(0, os.path.abspath(".."))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")
from icana import paths, checks, fits
print("ICDIR =", paths.ICDIR or "NOT SET (source env.sh)")

In [ ]:
from icana import tables_lt
cfg_f, lt_f, sns_f = tables_lt.read_fiber_s2_table(paths.S2_TABLE_FIBERS)
cfg_f

In [ ]:
lce_f, cols = tables_lt.fiber_lce_map(lt_f)
# The table covers one 20-degree sector. Bins outside the wedge (|phi| > 10 deg) are never used by IC,
# which rotates every point into the sector, so they are left out of the map.
lce_f["inside"] = np.abs(np.degrees(np.arctan2(lce_f.y, lce_f.x))) <= 10
print(f"{len(cols)} fiber sensors, {len(lce_f)} (x, y) bins, {lce_f.inside.sum()} inside the sector wedge")
w = lce_f[lce_f.inside]
sc = plt.scatter(w.x, w.y, c=w.lce * 100, s=12); plt.colorbar(sc, label="fiber LCE (%)")
plt.xlabel("x (mm)"); plt.ylabel("y (mm)"); plt.gca().set_aspect("equal")

In [ ]:
lt_p = pd.read_hdf(paths.S2_TABLE_PMTS, "/LT/LightTable")
print(pd.read_hdf(paths.S2_TABLE_PMTS, "/LT/Config"))
r_p = np.hypot(lt_p.x, lt_p.y)
sc = plt.scatter(lt_p.x, lt_p.y, c=lt_p.PmtR11410_total * 100, s=4); plt.colorbar(sc, label="PMT LCE (%)")
plt.gca().set_aspect("equal"); plt.xlabel("x (mm)"); plt.ylabel("y (mm)")

In [ ]:
# LCE evaluated exactly as detsim does, with IC's table classes (sum over sensors), along a radius
from invisible_cities.detsim.light_tables_c import LT_FIBERS, LT_PMT
ltf, ltp = LT_FIBERS(fname=paths.S2_TABLE_FIBERS), LT_PMT(fname=paths.S2_TABLE_PMTS)
def total(lt, x, y): return sum(np.sum(lt.get_values(x, y, i)) for i in range(lt.num_sensors))
rows = [dict(R=r, fibers=total(ltf, r, 0.1) * 100, pmts=total(ltp, r, 0.1) * 100) for r in [1, 10, 20, 30, 100, 200, 300, 400, 450, 480]]
lce_r = pd.DataFrame(rows); lce_r["ratio"] = lce_r.fibers / lce_r.pmts
print(lce_r.round(3).to_string(index=False))
c = lce_r[lce_r.R <= 30]
print(f"R <= 30 mm: fibers {c.fibers.mean():.3f} %, PMTs {c.pmts.mean():.3f} %, ratio {c.fibers.mean()/c.pmts.mean():.2f}")
print("Compare with the S2 ratio after detsim (03) and dorothea (05).")

### Notes
- The fiber table stores a detection probability per EL photon and per sensor (PDE included). detsim multiplies it by `el_gain` × number of electrons.
- The fiber table was built from the 500 × 100k `20240405_Next100_ie_s2` runs (`S2TableStatistics` = 5e7 ie⁻); the same runs as the 2024 offline table.
- Check that the two tables use the same EL gap and active radius (Config above).